# Fetch Zone List + Zone Polygons

**Purpose:** authenticate, pull the full list of mine-defined zones (load/dump/fuel/repair/etc.), then pull the polygon vertices for each zone with a recognized material type and save both to CSV.

Like `fetch_tracker_list.ipynb`, this is a metadata refresh — zone geometry rarely changes — not part of the recurring GPS data pull.

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd

from gps_lib import navixy_api, classify, io_utils

In [ ]:
hash_token = navixy_api.authenticate()
zone_list_df = navixy_api.get_zone_list(hash_token)
zone_list_df.shape

In [ ]:
# Save the raw zone list before classification (classification is a derived
# label applied at analysis time, not part of the source data).
out_path = io_utils.save_csv(zone_list_df, "zone_list.csv")
print("Saved:", out_path)

In [ ]:
filtered_df = classify.classify_zones(zone_list_df, drop_other=True)
print("Zones with a recognized material type:", filtered_df.shape[0])
filtered_df[["id", "label", "zone_material_type", "zone_load_type"]].head()

In [ ]:
zone_detail_all = []
for zone_id in filtered_df["id"]:
    detail = navixy_api.get_zone_detail_list(hash_token, zone_id)
    if detail is not None and not detail.empty:
        detail["zone_id"] = zone_id
        zone_detail_all.append(detail)
    else:
        print(f"No polygon data for zone {zone_id}")

print(f"Fetched details for {len(zone_detail_all)} zones.")

In [ ]:
zone_detail_all_df = pd.concat(zone_detail_all, ignore_index=True)
out_path = io_utils.save_csv(zone_detail_all_df, "zone_detail_all_df.csv")
print("Saved:", out_path)
zone_detail_all_df.head()